# TPU v5e-1 feasibility probe — SkySim PPO

This notebook is an **isolated hardware compatibility benchmark**, not the 40-million-step training run. It checks whether the selected Colab TPU is visible to PyTorch/XLA and compares a small synthetic neural-network workload on CPU vs TPU.

Important limits:
- The current project trains PPO with Stable-Baselines3 and calls the native C++ DroneCore simulator through `ctypes`.
- Selecting a TPU does **not** move that C++ physics simulation to TPU, and Stable-Baselines3 does not provide a drop-in TPU device setting.
- The benchmark below measures only neural-network compute. It cannot establish end-to-end PPO speedup.
- The production target remains one shared policy, 40 parallel environments, `N_STEPS=1000`, `BATCH_SIZE=4000`, and 40,000,000 environment transitions. This probe does not modify those settings or start that run.

## 1. Verify the selected Colab runtime
In Colab, choose **Runtime → Change runtime type → TPU v5e-1**, save, then run from the top.

In [ ]:
import os, platform, sys, time
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA visible:', torch.cuda.is_available())
except Exception as exc:
    print('PyTorch import failed:', repr(exc))

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print('torch_xla:', getattr(torch_xla, '__version__', 'version not exposed'))
    TPU_AVAILABLE = True
    TPU_DEVICE = xm.xla_device()
    print('XLA device:', TPU_DEVICE)
except Exception as exc:
    TPU_AVAILABLE = False
    TPU_DEVICE = None
    print('TPU / torch_xla is NOT ready:', repr(exc))
    print('Do not continue to the TPU benchmark until the Colab TPU runtime and compatible torch_xla are available.')

## 2. Benchmark a representative small MLP on CPU vs TPU
This is a controlled synthetic workload, not PPO. The first run includes XLA compilation effects, so we warm up before timing and explicitly synchronize the TPU.

In [ ]:
import torch
from torch import nn

if not TPU_AVAILABLE:
    raise RuntimeError('TPU is not available. Fix the runtime/XLA setup before running this cell.')

torch.manual_seed(42)
BATCH = 1024
INPUT_DIM = 64
HIDDEN_DIM = 256
OUTPUT_DIM = 4
WARMUP_STEPS = 20
MEASURE_STEPS = 100

def benchmark_mlp(device, use_xla=False):
    model = nn.Sequential(
        nn.Linear(INPUT_DIM, HIDDEN_DIM), nn.Tanh(),
        nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.Tanh(),
        nn.Linear(HIDDEN_DIM, OUTPUT_DIM)
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    x = torch.randn(BATCH, INPUT_DIM, device=device)
    y = torch.randn(BATCH, OUTPUT_DIM, device=device)

    def one_step():
        optimizer.zero_grad(set_to_none=True)
        loss = ((model(x) - y) ** 2).mean()
        loss.backward()
        optimizer.step()
        if use_xla:
            xm.mark_step()

    for _ in range(WARMUP_STEPS):
        one_step()
    if use_xla:
        xm.mark_step()
        xm.wait_device_ops()
    elif device.type == 'cuda':
        torch.cuda.synchronize(device)

    start = time.perf_counter()
    for _ in range(MEASURE_STEPS):
        one_step()
    if use_xla:
        xm.mark_step()
        xm.wait_device_ops()
    elif device.type == 'cuda':
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    return {'device': str(device), 'seconds': round(elapsed, 3),
            'steps_per_second': round(MEASURE_STEPS / elapsed, 2),
            'samples_per_second': round(MEASURE_STEPS * BATCH / elapsed, 1)}

cpu_result = benchmark_mlp(torch.device('cpu'))
tpu_result = benchmark_mlp(TPU_DEVICE, use_xla=True)
print('CPU:', cpu_result)
print('TPU:', tpu_result)
print('TPU / CPU step-throughput ratio:', round(tpu_result['steps_per_second'] / cpu_result['steps_per_second'], 2))

## 3. Interpretation and next step

Record the printed CPU and TPU metrics and any import/runtime errors.

- If TPU is unavailable or XLA setup fails, do not attempt the long training run.
- If TPU is faster for this MLP, that is only evidence that the accelerator can help neural-network math. It does **not** mean the current SB3 PPO pipeline can use TPU directly.
- Before moving the actual training, a separate implementation would need a TPU-compatible PPO path (for example, a carefully validated JAX/PyTorch-XLA implementation) and tests showing equivalent observations, actions, rewards, and update behavior. The existing C++ physics remains CPU-bound unless separately ported/vectorized.
- Do not change the production training constants as part of this experiment.